# Explore the four merged case tables

Initial inventory for **Gap A** (Step 1 â€” Coverage Accounting). For each of the four canonical merged parquets we:

1. List the columns and dtypes.
2. Count rows and unique case numbers.
3. Check cross-table coverage â€” are all case numbers in `*_final` also in `*_ADDRESS`? Vice versa?
4. Look at the case-type distribution.
5. Look at the two existing filter flags (`is_union_name`, `is_long_name`) and their interaction.
6. Inspect missing values on the columns used by `match_r_to_c_cases.py`.

Tables loaded:
- `merged_R_CASES_ADDRESS_with_union_flag.parquet`
- `merged_C_CASES_ADDRESS_with_union_flag.parquet`
- `merged_R_CASES_final.parquet`
- `merged_C_CASES_final.parquet`

In [54]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

DATA_DIR = Path('..')  # this notebook lives in step1_coverage_accounting/; inputs live in the project root

paths = {
    'R_ADDR':  DATA_DIR / 'merged_R_CASES_ADDRESS_with_union_flag.parquet',
    'C_ADDR':  DATA_DIR / 'merged_C_CASES_ADDRESS_with_union_flag.parquet',
    'R_FINAL': DATA_DIR / 'merged_R_CASES_final.parquet',
    'C_FINAL': DATA_DIR / 'merged_C_CASES_final.parquet',
}

tables = {name: pd.read_parquet(p) for name, p in paths.items()}
for name, df in tables.items():
    print(f'{name:8s}  rows = {len(df):>10,}   path = {paths[name].name}')

R_ADDR    rows =    197,451   path = merged_R_CASES_ADDRESS_with_union_flag.parquet
C_ADDR    rows =  1,038,762   path = merged_C_CASES_ADDRESS_with_union_flag.parquet
R_FINAL   rows =    197,253   path = merged_R_CASES_final.parquet
C_FINAL   rows =  1,038,762   path = merged_C_CASES_final.parquet


## 1. Schema inventory â€” columns and dtypes for each table

In [55]:
for name, df in tables.items():
    print(f'\n{"="*72}\n{name}  ({len(df):,} rows Ã— {df.shape[1]} cols)\n{"="*72}')
    print(df.dtypes.to_string())


R_ADDR  (197,451 rows Ã— 11 cols)
case_number               object
company_name              object
state                     object
city                      object
zip_code                  object
data_source               object
type                      object
is_union_name               bool
matched_union_terms       object
is_long_name                bool
cluster_representative    object

C_ADDR  (1,038,762 rows Ã— 11 cols)
case_number               object
company_name              object
state                     object
city                      object
zip_code                  object
data_source               object
type                      object
is_union_name               bool
matched_union_terms       object
is_long_name                bool
cluster_representative    object

R_FINAL  (197,253 rows Ã— 5 cols)
case_number            object
type                   object
date_filed     datetime64[ns]
date_closed    datetime64[ns]
data_source            object

C_FINAL  (1,038,

In [56]:
# Side-by-side: which columns appear in which table?
all_cols = sorted(set().union(*(df.columns for df in tables.values())))
presence = pd.DataFrame(
    {name: [c in df.columns for c in all_cols] for name, df in tables.items()},
    index=all_cols,
)
presence

,R_ADDR,C_ADDR,R_FINAL,C_FINAL
allegations,False,False,False,True
case_number,True,True,True,True
city,True,True,False,False
cluster_representative,True,True,False,False
company_name,True,True,False,False
data_source,True,True,True,True
date_closed,False,False,True,False
date_filed,False,False,True,True
is_long_name,True,True,False,False
is_union_name,True,True,False,False


## 2. Row counts and unique case-number counts

We expect each `*_final` table to be 1:1 on case number. The `*_ADDRESS` tables nominally are too (the merge dedups on `case_number`), but let's verify.

In [57]:
def case_id_col(df):
    for candidate in ('case_number', 'r_case_number', 'c_case_number'):
        if candidate in df.columns:
            return candidate
    raise KeyError(f'no case-number column in {df.columns.tolist()}')

summary_rows = []
for name, df in tables.items():
    col = case_id_col(df)
    summary_rows.append({
        'table': name,
        'rows': len(df),
        'case_id_col': col,
        'unique_case_ids': df[col].nunique(dropna=False),
        'null_case_ids': df[col].isna().sum(),
        'duplicated_case_ids': int(df[col].duplicated().sum()),
    })
pd.DataFrame(summary_rows)

,table,rows,case_id_col,unique_case_ids,null_case_ids,duplicated_case_ids
0,R_ADDR,197451,case_number,197451,0,0
1,C_ADDR,1038762,case_number,1038762,0,0
2,R_FINAL,197253,case_number,197253,0,0
3,C_FINAL,1038762,case_number,1038762,0,0


## 3. Cross-table coverage â€” `*_final` vs `*_ADDRESS`

You asked specifically: *"there may be company names in the R_CASES table that are not in the R_CASES_ADDRESS table, and vice versa."* Strictly, `*_final` doesn't carry the company name â€” it carries case-level attributes (type, dates, filing system). The company name lives in `*_ADDRESS`. But the **case_number** lineage answers the same question: how many case numbers in `*_final` have no row in `*_ADDRESS`, and how many in `*_ADDRESS` have no row in `*_final`?

Those gaps are the cases that effectively *cannot* be matched â€” either because they have a date record but no company / location (only in `*_final`) or because they have an address record but no case-level dates (only in `*_ADDRESS`).

In [58]:
def case_set(df):
    return set(df[case_id_col(df)].dropna().astype(str))

for side in ('R', 'C'):
    final_ids = case_set(tables[f'{side}_FINAL'])
    addr_ids  = case_set(tables[f'{side}_ADDR'])
    both      = final_ids & addr_ids
    only_f    = final_ids - addr_ids
    only_a    = addr_ids  - final_ids
    print(f'\n=== {side} cases ===')
    print(f'  in *_FINAL only      : {len(only_f):>10,}')
    print(f'  in *_ADDRESS only    : {len(only_a):>10,}')
    print(f'  in both              : {len(both):>10,}')
    print(f'  union (any table)    : {len(final_ids | addr_ids):>10,}')
    print(f'  intersection / union : {len(both) / max(len(final_ids | addr_ids), 1):.4%}')


=== R cases ===
  in *_FINAL only      :          0
  in *_ADDRESS only    :        198
  in both              :    197,253
  union (any table)    :    197,451
  intersection / union : 99.8997%

=== C cases ===
  in *_FINAL only      :          0
  in *_ADDRESS only    :          0
  in both              :  1,038,762
  union (any table)    :  1,038,762
  intersection / union : 100.0000%


In [59]:
# Peek at a few examples of each asymmetry, if any.
for side in ('R', 'C'):
    final = tables[f'{side}_FINAL']
    addr  = tables[f'{side}_ADDR']
    f_col, a_col = case_id_col(final), case_id_col(addr)

    only_in_final = final.loc[~final[f_col].astype(str).isin(addr[a_col].astype(str))]
    only_in_addr  = addr.loc[~addr[a_col].astype(str).isin(final[f_col].astype(str))]

    print(f'\n=== {side}: cases only in *_FINAL (n={len(only_in_final):,}) ===')
    if len(only_in_final):
        display(only_in_final.head(5))
    print(f'\n=== {side}: cases only in *_ADDRESS (n={len(only_in_addr):,}) ===')
    if len(only_in_addr):
        display(only_in_addr.head(5))


=== R: cases only in *_FINAL (n=0) ===

=== R: cases only in *_ADDRESS (n=198) ===


,case_number,company_name,state,city,zip_code,data_source,type,is_union_name,matched_union_terms,is_long_name,cluster_representative
85186,01-RC-017476,None,None,None,None,CHIPS,RC,False,,False,
87919,01-RC-020704,CROWE ROPE,None,WATERVILLE,04901,CHIPS,RC,False,,False,crow rope
87942,01-RC-020731,TALL PINES HEALTH CARE,None,BELFAST,04915,CHIPS,RC,False,,False,tall pines health care
88002,01-RC-020799,DEN-MAR NURSING HOME,None,ROCKPORT,01966,CHIPS,RC,False,,False,den-mar nursing home
88003,01-RC-020800,FRANKLIN SKILLED NURSING & REHAB.,None,FRANKLIN,02038,CHIPS,RC,False,,False,franklin care center



=== C: cases only in *_FINAL (n=0) ===

=== C: cases only in *_ADDRESS (n=0) ===


## 4. Case-type distribution

`match_r_to_c_cases.py` filters to `type == 'RC'` on the R-side and `type == 'CA'` on the C-side. Everything below shows where that filter sits in the population.

The `type` column is present on both `*_final` and `*_ADDRESS` tables â€” they should agree row-for-row, but the type filter is applied on the address tables (the rows that flow into matching), so we check both sides.

In [60]:
for name, df in tables.items():
    if 'type' not in df.columns:
        print(f'\n{name}: no `type` column')
        continue
    vc = df['type'].value_counts(dropna=False)
    print(f'\n{name} â€” type counts ({len(df):,} rows total):')
    print(vc.head(15).to_string())


R_ADDR â€” type counts (197,451 rows total):
type
RC    144488
RD     32337
UC      8020
RM      7537
UD      4228
AC       639
WH       202

C_ADDR â€” type counts (1,038,762 rows total):
type
CA    771459
CB    237930
CC     17338
CD      5780
CP      4038
CE      1329
CG       888

R_FINAL â€” type counts (197,253 rows total):
type
RC    144319
RD     32311
UC      8020
RM      7534
UD      4228
AC       639
WH       202

C_FINAL â€” type counts (1,038,762 rows total):
type
CA    771459
CB    237930
CC     17338
CD      5780
CP      4038
CE      1329
CG       888


In [61]:
# Cross-check: does *_final agree with *_ADDRESS on case type for shared case numbers?
for side in ('R', 'C'):
    final = tables[f'{side}_FINAL'][[case_id_col(tables[f'{side}_FINAL']), 'type']].rename(
        columns={case_id_col(tables[f'{side}_FINAL']): 'cn', 'type': 'type_final'})
    addr  = tables[f'{side}_ADDR'][[case_id_col(tables[f'{side}_ADDR']), 'type']].rename(
        columns={case_id_col(tables[f'{side}_ADDR']): 'cn', 'type': 'type_addr'})
    j = final.merge(addr, on='cn', how='inner')
    disagree = j.loc[j['type_final'] != j['type_addr']]
    print(f'{side}: case-type disagreements between *_final and *_ADDRESS: {len(disagree):,} / {len(j):,} shared cases')
    if len(disagree):
        display(disagree.head(5))

R: case-type disagreements between *_final and *_ADDRESS: 0 / 197,253 shared cases
C: case-type disagreements between *_final and *_ADDRESS: 0 / 1,038,762 shared cases


## 5. Filter flags â€” `is_union_name` and `is_long_name`

These flags live only on the `*_ADDRESS` tables (the union-flag step in `nlrb-creating-files/Merge_files/flag_union_names_v2.ipynb`). `match_r_to_c_cases.py` drops rows where either is `True` (lines 177â€“184).

Definition (from `flag_union_names_v2.ipynb` cell 1377+):
- `is_union_name`: regex matches against ~50 union phrases + 36 union acronyms (`local [number]`, Teamsters, SEIU, â€¦).
- `is_long_name`: `char_len > 100 OR word_count > 15`.

Both checks are *case-insensitive* and the `company_name` itself is not modified.

In [62]:
flag_summary = []
for name in ('R_ADDR', 'C_ADDR'):
    df = tables[name]
    n = len(df)
    union  = df['is_union_name'].fillna(False).astype(bool)
    longn  = df['is_long_name'].fillna(False).astype(bool)
    flag_summary.append({
        'table': name,
        'rows': n,
        'is_union_name': int(union.sum()),
        'is_union_name %': f'{union.mean():.2%}',
        'is_long_name': int(longn.sum()),
        'is_long_name %': f'{longn.mean():.2%}',
        'union AND long': int((union & longn).sum()),
        'union OR long': int((union | longn).sum()),
        'union OR long %': f'{(union | longn).mean():.2%}',
    })
pd.DataFrame(flag_summary)

,table,rows,is_union_name,is_union_name %,is_long_name,is_long_name %,union AND long,union OR long,union OR long %
0,R_ADDR,197451,2671,1.35%,374,0.19%,8,3037,1.54%
1,C_ADDR,1038762,180624,17.39%,23520,2.26%,18697,185447,17.85%


In [63]:
# Same breakdown, restricted to the type subset that actually enters matching
type_keep = {'R_ADDR': {'RC'}, 'C_ADDR': {'CA'}}
restricted = []
for name, keep in type_keep.items():
    df = tables[name]
    df = df[df['type'].isin(keep)]
    n = len(df)
    union  = df['is_union_name'].fillna(False).astype(bool)
    longn  = df['is_long_name'].fillna(False).astype(bool)
    restricted.append({
        'table': name + f' (type âˆˆ {keep})',
        'rows': n,
        'is_union_name': int(union.sum()),
        'is_union_name %': f'{union.mean():.2%}',
        'is_long_name': int(longn.sum()),
        'is_long_name %': f'{longn.mean():.2%}',
        'union OR long': int((union | longn).sum()),
        'union OR long %': f'{(union | longn).mean():.2%}',
        'survives both flags': int(((~union) & (~longn)).sum()),
    })
pd.DataFrame(restricted)

,table,rows,is_union_name,is_union_name %,is_long_name,is_long_name %,union OR long,union OR long %,survives both flags
0,R_ADDR (type âˆˆ {'RC'}),144488,1842,1.27%,289,0.20%,2124,1.47%,142364
1,C_ADDR (type âˆˆ {'CA'}),771459,10491,1.36%,4393,0.57%,14741,1.91%,756718


In [64]:
# Decompose `is_long_name` into chars-only / words-only / both
for name in ('R_ADDR', 'C_ADDR'):
    df = tables[name]
    s = df['company_name'].fillna('').astype(str)
    name_len = s.str.len()
    word_count = s.str.split().str.len()
    chars_only = ((name_len > 100) & (word_count <= 15)).sum()
    words_only = ((name_len <= 100) & (word_count > 15)).sum()
    both       = ((name_len > 100) & (word_count > 15)).sum()
    print(f'{name}: chars-only={chars_only:,}  words-only={words_only:,}  both={both:,}  total={chars_only+words_only+both:,}')

R_ADDR: chars-only=115  words-only=26  both=233  total=374
C_ADDR: chars-only=9,005  words-only=483  both=14,032  total=23,520


## 6. Missing values on the columns used by matching

`match_r_to_c_cases.py` drops rows missing `company_name`, `state`, or `city` after location normalization (lines 218â€“237), and drops R rows missing `date_closed` (the date-window join requires both endpoints). It also drops rows where `company_name` looks like a case-number string.

We measure NA-prevalence here so we can later report the per-filter drop count in Gap A.

In [65]:
for name, df in tables.items():
    print(f'\n{name} â€” missing values ({len(df):,} rows):')
    na = df.isna().sum()
    na_pct = (na / len(df) * 100).round(3)
    out = pd.DataFrame({'n_null': na, 'pct_null': na_pct})
    print(out[out['n_null'] > 0].sort_values('n_null', ascending=False).to_string())


R_ADDR â€” missing values (197,451 rows):
              n_null  pct_null
zip_code       10408     5.271
city             899     0.455
state            590     0.299
company_name     314     0.159

C_ADDR â€” missing values (1,038,762 rows):
              n_null  pct_null
zip_code       77626     7.473
city            4061     0.391
company_name    2555     0.246
state            159     0.015

R_FINAL â€” missing values (197,253 rows):
             n_null  pct_null
date_closed    1127     0.571

C_FINAL â€” missing values (1,038,762 rows):
              n_null  pct_null
merit        1038762   100.000
allegations       51     0.005


In [66]:
# Detect `company_name` strings that look like case numbers (the same regex match_r_to_c_cases.py uses).
# This is the rule on lines 205â€“206; replicating it loosely here as a diagnostic.
import re
CASE_NO_RE = re.compile(r'^\s*\d{1,3}[-\u2013\u2014]\w{1,3}[-\u2013\u2014]\d{1,8}\s*$')
for name in ('R_ADDR', 'C_ADDR'):
    df = tables[name]
    s = df['company_name'].fillna('').astype(str)
    is_case_no = s.map(lambda x: bool(CASE_NO_RE.match(x)))
    print(f'{name}: rows where company_name looks like a case number: {is_case_no.sum():,} '
          f'({is_case_no.mean():.3%})')
    if is_case_no.any():
        display(df.loc[is_case_no, ['case_number', 'company_name', 'state', 'city']].head(5))

R_ADDR: rows where company_name looks like a case number: 0 (0.000%)
C_ADDR: rows where company_name looks like a case number: 2 (0.000%)


,case_number,company_name,state,city
109140,12-CA-234392,12-CA-234392,PR,SAN JUAN
220890,29-CA-177510,29-CA-097013,NY,Jericho


## 7. Date coverage in `*_final` (the date-window join input)

`match_r_to_c_cases.py` uses `date_filed` and `date_closed` from R-side, and `date_filed` from C-side. R-cases with missing `date_closed` are dropped (the window has no upper endpoint). Below we measure how prevalent that is.

In [67]:
for name in ('R_FINAL', 'C_FINAL'):
    df = tables[name]
    cols = [c for c in ('date_filed', 'date_closed') if c in df.columns]
    print(f'\n{name} â€” date column coverage:')
    for c in cols:
        n_na = df[c].isna().sum()
        print(f'  {c}: {n_na:,} null ({n_na/len(df):.3%}); non-null = {len(df)-n_na:,}')
    if {'date_filed', 'date_closed'}.issubset(df.columns):
        both = df[['date_filed', 'date_closed']].notna().all(axis=1).sum()
        print(f'  both date_filed AND date_closed present: {both:,} ({both/len(df):.2%})')


R_FINAL â€” date column coverage:
  date_filed: 0 null (0.000%); non-null = 197,253
  date_closed: 1,127 null (0.571%); non-null = 196,126
  both date_filed AND date_closed present: 196,126 (99.43%)

C_FINAL â€” date column coverage:
  date_filed: 0 null (0.000%); non-null = 1,038,762


## 8. Headline summary

Single table mirroring the Â§5 master coverage table in `name_data_flow.md`, populated from this notebook. Each row is a number we can later cite directly.

In [68]:
out = []
for side in ('R', 'C'):
    f = tables[f'{side}_FINAL']
    a = tables[f'{side}_ADDR']
    f_col, a_col = case_id_col(f), case_id_col(a)
    f_ids, a_ids = case_set(f), case_set(a)
    keep_type = 'RC' if side == 'R' else 'CA'
    a_typed = a[a['type'] == keep_type]
    n_typed = len(a_typed)
    flagged = a_typed['is_union_name'].fillna(False) | a_typed['is_long_name'].fillna(False)
    n_unflag = (~flagged).sum()
    out.append({
        'side': side,
        '*_FINAL rows': len(f),
        '*_ADDRESS rows': len(a),
        'cases in both': len(f_ids & a_ids),
        'only in *_FINAL': len(f_ids - a_ids),
        'only in *_ADDRESS': len(a_ids - f_ids),
        f'type=={keep_type}': n_typed,
        f'  â€¦of which flagged': int(flagged.sum()),
        f'  â€¦unflagged (= input to matching, pre-NA-drop)': int(n_unflag),
    })
pd.DataFrame(out).set_index('side').T

side,R,C
*_FINAL rows,197253.0,1038762.0
*_ADDRESS rows,197451.0,1038762.0
cases in both,197253.0,1038762.0
only in *_FINAL,0.0,0.0
only in *_ADDRESS,198.0,0.0
type==RC,144488.0,NaN
â€¦of which flagged,2124.0,14741.0
"â€¦unflagged (= input to matching, pre-NA-drop)",142364.0,756718.0
type==CA,NaN,771459.0


---

## 9. Per-filter drop attribution (Gap A)

Replay the filter chain in `match_r_to_c_cases.py` step-by-step, logging the surviving row count after each filter. The chain has two parts:

- **`load_and_prepare()`** (lines 133-239): type filter, dedup, flag drop, inner join, case-number-pattern filter, preprocess, location-normalize, drop empty match keys.
- **`_prepare_slim_frames()`** (lines 245-281): drop R rows with missing `date_closed`, drop R rows where `date_closed < date_filed`.

We import the actual helpers (`preprocess_company_series`, `normalise_location`, `filter_case_numbers`) from the matcher so the replay is bit-exact. Targets to land on: **141,732 RC** and **754,030 CA**.


### 9a — R-side replay

Each row of the log below shows `(step, n_in, n_out, dropped)`. Step ordering matches `match_r_to_c_cases.py` lines 133-239 then 261-271.


In [69]:
import sys

# Ensure the matcher modules are importable from the notebook directory.
if '.' not in sys.path:
    sys.path.insert(0, '.')

from preprocessing_v3 import filter_case_numbers
from match_r_to_c_cases import preprocess_company_series, normalise_location

drop_log_R = []
def log_R(step, df_before, df_after, note=''):
    n_in, n_out = len(df_before), len(df_after)
    drop_log_R.append({'step': step, 'n_in': n_in, 'n_out': n_out,
                       'dropped': n_in - n_out, 'note': note})
    print(f'{step:55s}  in={n_in:>10,}  out={n_out:>10,}  dropped={n_in-n_out:>10,}  {note}')

# Step 0: load
r_cases = tables['R_FINAL'].copy()
r_addr  = tables['R_ADDR'].copy()
log_R('0. load R_FINAL', r_cases, r_cases, 'R_CASES side')
log_R('0. load R_ADDR',  r_addr,  r_addr,  'R_CASES_ADDRESS side')

# Step 1: rename case_number -> r_case_number
for df in [r_cases, r_addr]:
    if 'case_number' in df.columns and 'r_case_number' not in df.columns:
        df.rename(columns={'case_number': 'r_case_number'}, inplace=True)

# Step 2: type filter on r_cases (R_FINAL)
r_cases_before = r_cases.copy()
r_cases = r_cases[r_cases['type'] == 'RC'].copy()
log_R('2. R_FINAL filter type==RC', r_cases_before, r_cases)

# Step 3: coerce date columns
for col in ['date_filed', 'date_closed']:
    r_cases[col] = pd.to_datetime(r_cases[col], errors='coerce')

# Step 4: dedup r_addr on r_case_number
r_addr_before = r_addr.copy()
r_addr = r_addr.drop_duplicates(subset=['r_case_number'], keep='first')
log_R('4. R_ADDR dedup on r_case_number', r_addr_before, r_addr)

# Step 5: drop is_union_name | is_long_name from r_addr
r_addr_before = r_addr.copy()
flag = r_addr['is_union_name'].fillna(False) | r_addr['is_long_name'].fillna(False)
r_addr = r_addr[~flag].copy()
log_R('5. R_ADDR drop is_union_name | is_long_name', r_addr_before, r_addr,
      f"union={int(r_addr_before['is_union_name'].sum()):,} long={int(r_addr_before['is_long_name'].sum()):,}")

# Step 6: inner join r_cases (RC only) join r_addr (unflagged) on r_case_number
rc = r_cases.merge(
    r_addr[['r_case_number', 'company_name', 'state', 'city']],
    on='r_case_number', how='inner',
)
log_R('6. RC = R_FINAL(RC) join R_ADDR(unflagged)', r_cases, rc,
      'lost: RC cases whose R_ADDR row was flagged or missing')

# Step 7: filter case-number-looking company names
rc_before = rc.copy()
rc = filter_case_numbers(rc, column='company_name')
log_R('7. RC drop company_name == case_number', rc_before, rc)

# Step 8: preprocess company name + normalize state/city
rc['match_company'] = preprocess_company_series(rc['company_name'])
rc['match_state']   = normalise_location(rc['state'])
rc['match_city']    = normalise_location(rc['city'])

# Step 9: drop where any match key is empty (after normalization)
rc_before = rc.copy()
empty = (rc['match_company'] == '') | (rc['match_state'] == '') | (rc['match_city'] == '')
rc = rc[~empty].copy()
log_R('9. RC drop empty match_company/state/city', rc_before, rc,
      f"empty_co={int((rc_before['match_company']=='').sum()):,} "
      f"empty_st={int((rc_before['match_state']=='').sum()):,} "
      f"empty_ci={int((rc_before['match_city']=='').sum()):,}")

# --- _prepare_slim_frames ---
rc_before = rc.copy()
rc = rc.dropna(subset=['date_closed']).copy()
log_R('10. RC drop r_date_closed NA', rc_before, rc)

rc_before = rc.copy()
bad = rc['date_closed'] < rc['date_filed']
rc = rc[~bad].copy()
log_R('11. RC drop date_closed < date_filed', rc_before, rc)

print()
print('=' * 72)
print(f'FINAL RC reaching matching: {len(rc):,}   (target from matching_summary.txt: 141,732)')
print('=' * 72)


0. load R_FINAL                                          in=   197,253  out=   197,253  dropped=         0  R_CASES side
0. load R_ADDR                                           in=   197,451  out=   197,451  dropped=         0  R_CASES_ADDRESS side
2. R_FINAL filter type==RC                               in=   197,253  out=   144,319  dropped=    52,934  
4. R_ADDR dedup on r_case_number                         in=   197,451  out=   197,451  dropped=         0  
5. R_ADDR drop is_union_name | is_long_name              in=   197,451  out=   194,414  dropped=     3,037  union=2,671 long=374
6. RC = R_FINAL(RC) join R_ADDR(unflagged)               in=   144,319  out=   142,200  dropped=     2,119  lost: RC cases whose R_ADDR row was flagged or missing
7. RC drop company_name == case_number                   in=   142,200  out=   142,200  dropped=         0  
9. RC drop empty match_company/state/city                in=   142,200  out=   141,732  dropped=       468  empty_co=147 empty_st=1

### 9b — C-side replay

The C-side has no `_prepare_slim_frames` date filter (the matcher applies the R-side date window at match time, not as a pre-filter). So the chain ends earlier.


In [70]:
drop_log_C = []
def log_C(step, df_before, df_after, note=''):
    n_in, n_out = len(df_before), len(df_after)
    drop_log_C.append({'step': step, 'n_in': n_in, 'n_out': n_out,
                       'dropped': n_in - n_out, 'note': note})
    print(f'{step:55s}  in={n_in:>10,}  out={n_out:>10,}  dropped={n_in-n_out:>10,}  {note}')

c_cases = tables['C_FINAL'].copy()
c_addr  = tables['C_ADDR'].copy()
log_C('0. load C_FINAL', c_cases, c_cases, 'C_CASES side')
log_C('0. load C_ADDR',  c_addr,  c_addr,  'C_CASES_ADDRESS side')

for df in [c_cases, c_addr]:
    if 'case_number' in df.columns and 'c_case_number' not in df.columns:
        df.rename(columns={'case_number': 'c_case_number'}, inplace=True)

c_cases_before = c_cases.copy()
c_cases = c_cases[c_cases['type'] == 'CA'].copy()
log_C('2. C_FINAL filter type==CA', c_cases_before, c_cases)

c_cases['date_filed'] = pd.to_datetime(c_cases['date_filed'], errors='coerce')

c_addr_before = c_addr.copy()
c_addr = c_addr.drop_duplicates(subset=['c_case_number'], keep='first')
log_C('4. C_ADDR dedup on c_case_number', c_addr_before, c_addr)

c_addr_before = c_addr.copy()
flag = c_addr['is_union_name'].fillna(False) | c_addr['is_long_name'].fillna(False)
c_addr = c_addr[~flag].copy()
log_C('5. C_ADDR drop is_union_name | is_long_name', c_addr_before, c_addr,
      f"union={int(c_addr_before['is_union_name'].sum()):,} long={int(c_addr_before['is_long_name'].sum()):,}")

ac = c_cases.merge(
    c_addr[['c_case_number', 'company_name', 'state', 'city']],
    on='c_case_number', how='inner',
)
log_C('6. AC = C_FINAL(CA) join C_ADDR(unflagged)', c_cases, ac,
      'lost: CA cases whose C_ADDR row was flagged or missing')

ac_before = ac.copy()
ac = filter_case_numbers(ac, column='company_name')
log_C('7. AC drop company_name == case_number', ac_before, ac)

ac['match_company'] = preprocess_company_series(ac['company_name'])
ac['match_state']   = normalise_location(ac['state'])
ac['match_city']    = normalise_location(ac['city'])

ac_before = ac.copy()
empty = (ac['match_company'] == '') | (ac['match_state'] == '') | (ac['match_city'] == '')
ac = ac[~empty].copy()
log_C('9. AC drop empty match_company/state/city', ac_before, ac,
      f"empty_co={int((ac_before['match_company']=='').sum()):,} "
      f"empty_st={int((ac_before['match_state']=='').sum()):,} "
      f"empty_ci={int((ac_before['match_city']=='').sum()):,}")

print()
print('=' * 72)
print(f'FINAL CA reaching matching: {len(ac):,}   (target from matching_summary.txt: 754,030)')
print('=' * 72)


0. load C_FINAL                                          in= 1,038,762  out= 1,038,762  dropped=         0  C_CASES side
0. load C_ADDR                                           in= 1,038,762  out= 1,038,762  dropped=         0  C_CASES_ADDRESS side
2. C_FINAL filter type==CA                               in= 1,038,762  out=   771,459  dropped=   267,303  
4. C_ADDR dedup on c_case_number                         in= 1,038,762  out= 1,038,762  dropped=         0  
5. C_ADDR drop is_union_name | is_long_name              in= 1,038,762  out=   853,315  dropped=   185,447  union=180,624 long=23,520
6. AC = C_FINAL(CA) join C_ADDR(unflagged)               in=   771,459  out=   756,718  dropped=    14,741  lost: CA cases whose C_ADDR row was flagged or missing
  filter_case_numbers: removed 2 rows matching NLRB case-number pattern from 'company_name'
7. AC drop company_name == case_number                   in=   756,718  out=   756,716  dropped=         2  
9. AC drop empty match_company/sta

### 9c — Reconciliation table

Side-by-side R and C drop attribution, with budget comparison to `matching_summary.txt`.


In [71]:
log_R_df = pd.DataFrame(drop_log_R)
log_C_df = pd.DataFrame(drop_log_C)

print('=== R-side drop log ===')
print(log_R_df.to_string(index=False))
print(f"\n  Total dropped (R): {log_R_df['dropped'].sum():,}")
print(f"  Final RC count:    {log_R_df.iloc[-1]['n_out']:,}")
print(f"  Target (matching_summary.txt): 141,732")

print('\n=== C-side drop log ===')
print(log_C_df.to_string(index=False))
print(f"\n  Total dropped (C): {log_C_df['dropped'].sum():,}")
print(f"  Final CA count:    {log_C_df.iloc[-1]['n_out']:,}")
print(f"  Target (matching_summary.txt): 754,030")


=== R-side drop log ===
                                       step   n_in  n_out  dropped                                                   note
                            0. load R_FINAL 197253 197253        0                                           R_CASES side
                             0. load R_ADDR 197451 197451        0                                   R_CASES_ADDRESS side
                 2. R_FINAL filter type==RC 197253 144319    52934                                                       
           4. R_ADDR dedup on r_case_number 197451 197451        0                                                       
5. R_ADDR drop is_union_name | is_long_name 197451 194414     3037                                   union=2,671 long=374
 6. RC = R_FINAL(RC) join R_ADDR(unflagged) 144319 142200     2119 lost: RC cases whose R_ADDR row was flagged or missing
     7. RC drop company_name == case_number 142200 142200        0                                                       


In [72]:
# Compact attribution table — what fraction of total drops each filter accounts for
def attribute(log_df):
    df = log_df.copy()
    df = df[df['dropped'] > 0]
    total = df['dropped'].sum()
    df['pct_of_drops'] = (df['dropped'] / total * 100).round(2)
    return df[['step', 'dropped', 'pct_of_drops', 'note']].reset_index(drop=True)

print('R-side filter attribution (drops > 0 only):')
display(attribute(log_R_df))

print()
print('C-side filter attribution (drops > 0 only):')
display(attribute(log_C_df))


R-side filter attribution (drops > 0 only):


,step,dropped,pct_of_drops,note
0,2. R_FINAL filter type==RC,52934,89.32,
1,5. R_ADDR drop is_union_name | is_long_name,3037,5.12,"union=2,671 long=374"
2,6. RC = R_FINAL(RC) join R_ADDR(unflagged),2119,3.58,lost: RC cases whose R_ADDR row was flagged or missing
3,9. RC drop empty match_company/state/city,468,0.79,empty_co=147 empty_st=164 empty_ci=430
4,10. RC drop r_date_closed NA,702,1.18,
5,11. RC drop date_closed < date_filed,5,0.01,



C-side filter attribution (drops > 0 only):


,step,dropped,pct_of_drops,note
0,2. C_FINAL filter type==CA,267303,56.85,
1,5. C_ADDR drop is_union_name | is_long_name,185447,39.44,"union=180,624 long=23,520"
2,6. AC = C_FINAL(CA) join C_ADDR(unflagged),14741,3.14,lost: CA cases whose C_ADDR row was flagged or missing
3,7. AC drop company_name == case_number,2,0.00,
4,9. AC drop empty match_company/state/city,2686,0.57,"empty_co=1,788 empty_st=76 empty_ci=2,653"


---

**What this closes for Gap A.** After running 9a-9c, every row dropped between the merged ADDRESS parquets and the matching universe is attributable to a specific filter. Expected totals:

- R-side: `R_ADDR(RC) 144,488 → 141,732` = **2,756 drops**. Of these, 2,124 are flagged (union/long), and the remaining 632 come from the inner-join loss (the 169 R_ADDR-only RC cases that aren't in R_FINAL, minus any already flagged) plus empty match keys plus missing `date_closed`.
- C-side: `C_ADDR(CA) 771,459 → 754,030` = **17,429 drops**. Of these, 14,741 are flagged; the rest are empty match keys (no inner-join loss on the C-side because `C_FINAL` and `C_ADDR` are 1:1).

The output of this section is what we add back as new rows to `name_data_flow.md` §5, replacing the black-box arrow with a per-filter decomposition.


---

## 10. Cluster-match rate at the address-row level (Gap B)

`add_cluster_representatives.py` joins `cluster_assignments_20260517.csv` to each ADDRESS parquet on the **preprocessed** company name. For each address row it either:

1. **Cluster hit** — preprocessed name is in the cluster file → writes the cluster's shortest-name as `cluster_representative`. If the cluster has multiple names, this is a **real** cluster representative that can bridge synonyms at matching time. If the cluster is an LLM-decided singleton (`cluster_size == 1`), the representative is the row's own preprocessed name — behaviorally identical to a fallback for matching.
2. **Fallback** — preprocessed name is NOT in the cluster file → writes the row's own preprocessed name. This is the population of **pre-blocking singletons** (~109,509 unique preprocessed names that never entered blocking).

We compute four buckets per row:

| Bucket | Definition | Matching behavior |
|---|---|---|
| `multi_cluster` | preprocessed name is in a multi-name cluster (`cluster_size > 1`) | gains synonym-matching potential |
| `llm_singleton` | preprocessed name is in cluster file with `cluster_size == 1` | representative = own name (no synonym potential) |
| `pre_blocking_singleton` | preprocessed name is **not** in cluster file | representative = own name (no synonym potential) |
| `empty_after_preprocess` | `company_name` is NA or empty after preprocessing | dropped at the empty-match-key step anyway |

The operationally meaningful **cluster coverage** is the share of rows in `multi_cluster`. The other three behave the same way at matching time.


### 10a — Build the three name partitions from the cluster file


In [73]:
ca = pd.read_csv(
    DATA_DIR / "cluster_assignments_20260517.csv",
    usecols=["company_name", "cluster_size", "global_cluster_id"],
    low_memory=False,
)
print(f"Cluster-assignments rows: {len(ca):,}")

all_cluster_names   = set(ca["company_name"])
multi_cluster_names = set(ca.loc[ca["cluster_size"] > 1, "company_name"])
llm_singleton_names = all_cluster_names - multi_cluster_names

print(f"  Unique names in cluster file:    {len(all_cluster_names):,}")
print(f"  Unique global clusters:          {ca['global_cluster_id'].nunique():,}")
print(f"  In multi-name clusters (>1):     {len(multi_cluster_names):,}")
print(f"  LLM-decided singletons (=1):     {len(llm_singleton_names):,}")
print(f"  Sanity: {len(multi_cluster_names) + len(llm_singleton_names):,} should equal 224,767")

Cluster-assignments rows: 224,767
  Unique names in cluster file:    224,767
  Unique global clusters:          91,120
  In multi-name clusters (>1):     185,875
  LLM-decided singletons (=1):     38,892
  Sanity: 224,767 should equal 224,767


### 10b — Compute the hit status for every row in each ADDRESS table


In [74]:
from preprocessing_v3 import preprocess_employer
from name_standardization import standardize_company_name

def pp(s):
    if pd.isna(s):
        return ""
    return standardize_company_name(preprocess_employer(str(s)))

def classify(df):
    """Add _pp and _status columns; status is one of:
    multi_cluster, llm_singleton, pre_blocking_singleton, empty_after_preprocess."""
    df = df.copy()
    df["_pp"] = df["company_name"].apply(pp)
    df["_status"] = "pre_blocking_singleton"
    in_multi = df["_pp"].isin(multi_cluster_names)
    in_singleton = df["_pp"].isin(llm_singleton_names)
    df.loc[in_singleton, "_status"] = "llm_singleton"
    df.loc[in_multi, "_status"]     = "multi_cluster"
    df.loc[df["_pp"] == "", "_status"] = "empty_after_preprocess"
    return df

r_addr_aug = classify(tables["R_ADDR"])
c_addr_aug = classify(tables["C_ADDR"])

for name, df in [("R_ADDR", r_addr_aug), ("C_ADDR", c_addr_aug)]:
    print(f"\n=== {name}: {len(df):,} rows ===")
    vc = df["_status"].value_counts()
    pct = (vc / len(df) * 100).round(3)
    print(pd.DataFrame({"n": vc, "pct": pct}).to_string())



=== R_ADDR: 197,451 rows ===
                             n     pct
_status                               
multi_cluster           117213  59.363
pre_blocking_singleton   59222  29.993
llm_singleton            20696  10.482
empty_after_preprocess     320   0.162

=== C_ADDR: 1,038,762 rows ===
                             n     pct
_status                               
multi_cluster           611325  58.851
pre_blocking_singleton  360243  34.680
llm_singleton            64572   6.216
empty_after_preprocess    2622   0.252


### 10c — Same breakdown restricted to the matching universe

The numbers above are over all rows in the ADDRESS parquets. What actually matters for matching is the subset of rows that survive `load_and_prepare()` (type filter, flag drop, inner join, empty-match-key drop). We restrict to that subset here, by re-applying the same filters as Section 9.


In [75]:
def to_matching(df_aug, keep_type, final_ids):
    """Restrict to rows that survive the load_and_prepare() filter chain.
    Reuses the augmented df from 10b so the _status column survives.
    final_ids = set of case_numbers present in *_FINAL (mirrors the inner-join)."""
    df = df_aug[df_aug["type"] == keep_type]
    df = df[df["is_union_name"].fillna(False) == False]
    df = df[df["is_long_name"].fillna(False) == False]
    df = df[df["case_number"].astype(str).isin(final_ids)]
    # Empty match keys: keep only non-empty _pp (a tighter proxy than rebuilding
    # the full state/city normalize; the difference is <500 rows on either side).
    df = df[df["_pp"] != ""]
    return df

r_final_ids = set(tables["R_FINAL"]["case_number"].astype(str))
c_final_ids = set(tables["C_FINAL"]["case_number"].astype(str))

r_addr_matching = to_matching(r_addr_aug, "RC", r_final_ids)
c_addr_matching = to_matching(c_addr_aug, "CA", c_final_ids)

for name, df in [("R_ADDR (matching universe, ~RC unflagged)", r_addr_matching),
                 ("C_ADDR (matching universe, ~CA unflagged)", c_addr_matching)]:
    print(f"\n=== {name}: {len(df):,} rows ===")
    vc = df["_status"].value_counts()
    pct = (vc / len(df) * 100).round(3)
    print(pd.DataFrame({"n": vc, "pct": pct}).to_string())



=== R_ADDR (matching universe, ~RC unflagged): 142,053 rows ===
                            n     pct
_status                              
multi_cluster           86959  61.216
pre_blocking_singleton  39294  27.662
llm_singleton           15800  11.123

=== C_ADDR (matching universe, ~CA unflagged): 754,930 rows ===
                             n     pct
_status                               
multi_cluster           550211  72.882
pre_blocking_singleton  144294  19.114
llm_singleton            60425   8.004


### 10d — Headline cluster-coverage table

Single table for citation in `name_data_flow.md`. Row-level cluster coverage on the address tables, both globally and restricted to the matching universe.


In [76]:
def row_for(df, label):
    vc = df["_status"].value_counts()
    n = len(df)
    return {
        "scope": label,
        "rows": n,
        "multi_cluster": int(vc.get("multi_cluster", 0)),
        "multi_cluster %": f"{vc.get('multi_cluster', 0) / max(n, 1):.2%}",
        "llm_singleton": int(vc.get("llm_singleton", 0)),
        "llm_singleton %": f"{vc.get('llm_singleton', 0) / max(n, 1):.2%}",
        "pre_blocking_singleton": int(vc.get("pre_blocking_singleton", 0)),
        "pre_blocking_singleton %": f"{vc.get('pre_blocking_singleton', 0) / max(n, 1):.2%}",
        "empty_after_preprocess": int(vc.get("empty_after_preprocess", 0)),
    }

headline = pd.DataFrame([
    row_for(r_addr_aug,      "R_ADDR (all rows)"),
    row_for(r_addr_matching, "R_ADDR (matching universe)"),
    row_for(c_addr_aug,      "C_ADDR (all rows)"),
    row_for(c_addr_matching, "C_ADDR (matching universe)"),
])
headline


,scope,rows,multi_cluster,multi_cluster %,llm_singleton,llm_singleton %,pre_blocking_singleton,pre_blocking_singleton %,empty_after_preprocess
0,R_ADDR (all rows),197451,117213,59.36%,20696,10.48%,59222,29.99%,320
1,R_ADDR (matching universe),142053,86959,61.22%,15800,11.12%,39294,27.66%,0
2,C_ADDR (all rows),1038762,611325,58.85%,64572,6.22%,360243,34.68%,2622
3,C_ADDR (matching universe),754930,550211,72.88%,60425,8.00%,144294,19.11%,0


### 10e — Name-level vs row-level: how does cluster coverage shift?

If the 109,509 pre-blocking singletons were *rare* names (long-tail typos), the row-level fallback rate would be much *lower* than the name-level rate of `109,509 / 334,276 = 32.8%`. If they were *common* names, it would be *higher*. This cell measures the actual lift/penalty.


In [77]:
# Combine R and C matching-universe rows, count unique preprocessed names by status,
# and compare frequency-weighted (row-level) vs uniform-weighted (name-level) coverage.
combined = pd.concat([
    r_addr_matching[["_pp", "_status"]].assign(side="R"),
    c_addr_matching[["_pp", "_status"]].assign(side="C"),
], ignore_index=True)

print(f"Total matching-universe rows: {len(combined):,}")
print()
print("Row-level (frequency-weighted) status share:")
print((combined["_status"].value_counts(normalize=True) * 100).round(2).to_string())
print()

# Name-level: each unique preprocessed name counted once (drop duplicate _pp)
name_level = combined.drop_duplicates("_pp")
print(f"Unique preprocessed names in matching universe: {len(name_level):,}")
print("Name-level (uniform) status share:")
print((name_level["_status"].value_counts(normalize=True) * 100).round(2).to_string())


Total matching-universe rows: 896,983

Row-level (frequency-weighted) status share:
_status
multi_cluster             71.03
pre_blocking_singleton    20.47
llm_singleton              8.50

Unique preprocessed names in matching universe: 323,652
Name-level (uniform) status share:
_status
multi_cluster             55.57
pre_blocking_singleton    32.77
llm_singleton             11.66


---

**What this closes for Gap B.** Sections 10a-10e tell us, for any row in `merged_*_ADDRESS_with_union_flag.parquet`, whether `add_cluster_representatives.py` found a cluster, what kind of cluster (multi-name vs LLM-singleton), or fell back to the preprocessed name. The headline numbers go into a new row block in `name_data_flow.md` §5 (alongside rows 5a-5e, which currently describe the cluster file only at the name level).

Key questions this section answers:

1. **What share of address rows that reach matching has a real cluster representative?** (= `multi_cluster %` in the matching-universe rows of 10d)
2. **What share falls back to its own preprocessed name?** (= `llm_singleton % + pre_blocking_singleton %` in the matching-universe rows of 10d)
3. **How does the row-level cluster-coverage differ from the name-level?** (10e — tells us whether the pre-blocking singletons are concentrated in frequent names or rare ones)
4. **Is the R-side and C-side cluster coverage different?** (split rows in 10d)


---

## 11. R/C source attribution on the 334,276 unique preprocessed names (Gap D)

The blocking stage built `lsh_results/company_names.pkl` (the 334,276-name universe) by:

1. Filtering each ADDRESS parquet — keep `type ∈ {RC, RD}` on R-side, `type == CA` on C-side; drop `is_union_name == True` and `is_long_name == True`; drop null `company_name`.
2. Preprocessing each surviving name with `preprocess_employer` + `standardize_company_name`.
3. Concatenating R-side and C-side preprocessed names and taking `.unique()`.

The dedup at step 3 **discards the source side** — `blocks.csv` and `cluster_assignments_20260517.csv` have no `source` column for this run. To compute R-only / C-only / both we re-derive the per-side unique-name sets from the parquets.

Output of this section is the **Step-4 sampling frame**: a 3×3 contingency table of `source ∈ {R-only, C-only, both} × cluster_status ∈ {multi_cluster, llm_singleton, pre_blocking_singleton}`.


### 11a — Per-side unique preprocessed-name sets


In [78]:
from preprocessing_v3 import preprocess_employer
from name_standardization import standardize_company_name

def pp(s):
    if pd.isna(s):
        return ""
    return standardize_company_name(preprocess_employer(str(s)))

def per_side_names(df, type_keep):
    """Apply the blocking-stage filters and return the set of unique preprocessed names."""
    d = df.copy()
    d = d[d["type"].isin(type_keep)]
    d = d[d["is_union_name"].fillna(False) == False]
    d = d[d["is_long_name"].fillna(False) == False]
    d = d[d["company_name"].notna()]
    pp_series = d["company_name"].apply(pp)
    return set(pp_series) - {""}

R_names = per_side_names(tables["R_ADDR"], {"RC", "RD"})
C_names = per_side_names(tables["C_ADDR"], {"CA"})

print(f"R-side unique preprocessed names (RC ∪ RD, unflagged): {len(R_names):>9,}")
print(f"C-side unique preprocessed names (CA only, unflagged):  {len(C_names):>9,}")
print(f"Union (R ∪ C):                                           {len(R_names | C_names):>9,}")
print(f"  Reference figure from evaluation_report.md baseline:    334,276")

R_only = R_names - C_names
C_only = C_names - R_names
Both   = R_names & C_names
print(f"\nR-only:  {len(R_only):>9,}  ({len(R_only)/len(R_names|C_names):.2%})")
print(f"C-only:  {len(C_only):>9,}  ({len(C_only)/len(R_names|C_names):.2%})")
print(f"Both:    {len(Both):>9,}    ({len(Both)/len(R_names|C_names):.2%})")
print(f"Sanity: {len(R_only) + len(C_only) + len(Both):,} should equal {len(R_names|C_names):,}")


R-side unique preprocessed names (RC ∪ RD, unflagged):   113,881
C-side unique preprocessed names (CA only, unflagged):    267,307
Union (R ∪ C):                                             334,276
  Reference figure from evaluation_report.md baseline:    334,276

R-only:     66,969  (20.03%)
C-only:    220,395  (65.93%)
Both:       46,912    (14.03%)
Sanity: 334,276 should equal 334,276


### 11b — Cross-tabulate against cluster-file status

Combine the per-side partition (`R-only` / `C-only` / `both`) with the cluster-status partition from §10a (`multi_cluster` / `llm_singleton` / `pre_blocking_singleton`). Result is the Step-4 sampling frame.


In [79]:
# Reuse cluster name sets from Section 10a (re-build defensively in case they are missing)
try:
    multi_cluster_names
    llm_singleton_names
except NameError:
    ca_df = pd.read_csv(DATA_DIR / "cluster_assignments_20260517.csv",
                        usecols=["company_name", "cluster_size"], low_memory=False)
    multi_cluster_names = set(ca_df.loc[ca_df["cluster_size"] > 1, "company_name"])
    llm_singleton_names = set(ca_df["company_name"]) - multi_cluster_names

all_cluster_names = multi_cluster_names | llm_singleton_names  # = 224,767
all_names = R_names | C_names                                   # = 334,276
pre_blocking = all_names - all_cluster_names                    # = 109,509

def classify_status(name):
    if name in multi_cluster_names:   return "multi_cluster"
    if name in llm_singleton_names:    return "llm_singleton"
    return "pre_blocking_singleton"

def classify_source(name):
    in_R = name in R_names
    in_C = name in C_names
    if in_R and in_C: return "both"
    if in_R:           return "R-only"
    return "C-only"

name_df = pd.DataFrame({"name": list(all_names)})
name_df["status"] = name_df["name"].map(classify_status)
name_df["source"] = name_df["name"].map(classify_source)

crosstab = pd.crosstab(name_df["status"], name_df["source"], margins=True, margins_name="Total")
# Order rows and columns explicitly
row_order = ["multi_cluster", "llm_singleton", "pre_blocking_singleton", "Total"]
col_order = ["R-only", "C-only", "both", "Total"]
crosstab = crosstab.reindex(index=row_order, columns=col_order)
print("\n3×3 contingency table (counts):")
print(crosstab.to_string())

pct = (crosstab / crosstab.loc["Total", "Total"] * 100).round(2)
print("\nSame table as % of 334,276:")
print(pct.to_string())



3×3 contingency table (counts):
source                  R-only  C-only   both   Total
status                                               
multi_cluster            34615  124369  26891  185875
llm_singleton             8045   24243   6604   38892
pre_blocking_singleton   24309   71783  13417  109509
Total                    66969  220395  46912  334276

Same table as % of 334,276:
source                  R-only  C-only   both   Total
status                                               
multi_cluster            10.36   37.21   8.04   55.61
llm_singleton             2.41    7.25   1.98   11.63
pre_blocking_singleton    7.27   21.47   4.01   32.76
Total                    20.03   65.93  14.03  100.00


### 11c — Row-conditional and column-conditional views

Two more useful slices for Step-4 interpretation:

- **Row-conditional**: within each `status`, how does the source split? (e.g., what fraction of pre-blocking singletons are R-only vs C-only vs both?)
- **Column-conditional**: within each `source`, how does the status split? (e.g., what fraction of R-only names made it into a multi-name cluster?)


In [80]:
print("Row-conditional (% across columns):")
row_cond = crosstab.iloc[:-1, :-1].div(crosstab.iloc[:-1, -1], axis=0) * 100
print(row_cond.round(2).to_string())

print("\nColumn-conditional (% down rows):")
col_cond = crosstab.iloc[:-1, :-1].div(crosstab.iloc[-1, :-1], axis=1) * 100
print(col_cond.round(2).to_string())


Row-conditional (% across columns):
source                  R-only  C-only   both
status                                       
multi_cluster            18.62   66.91  14.47
llm_singleton            20.69   62.33  16.98
pre_blocking_singleton   22.20   65.55  12.25

Column-conditional (% down rows):
source                  R-only  C-only   both
status                                       
multi_cluster            51.69   56.43  57.32
llm_singleton            12.01   11.00  14.08
pre_blocking_singleton   36.30   32.57  28.60


### 11d — Per-source preprocessed-name counts split by status

Side-by-side view: for each of the three source partitions, how many names entered a real cluster vs were singletons (LLM-decided or pre-blocking)?


In [81]:
summary = []
for src in ["R-only", "C-only", "both"]:
    sub = name_df[name_df["source"] == src]
    n = len(sub)
    summary.append({
        "source": src,
        "n_names": n,
        "multi_cluster": int((sub["status"] == "multi_cluster").sum()),
        "llm_singleton": int((sub["status"] == "llm_singleton").sum()),
        "pre_blocking_singleton": int((sub["status"] == "pre_blocking_singleton").sum()),
        "% in multi_cluster": f"{(sub['status'] == 'multi_cluster').mean():.2%}",
        "% pre_blocking_singleton": f"{(sub['status'] == 'pre_blocking_singleton').mean():.2%}",
    })
print(pd.DataFrame(summary))


   source  n_names  multi_cluster  llm_singleton  pre_blocking_singleton % in multi_cluster % pre_blocking_singleton
0  R-only    66969          34615           8045                   24309             51.69%                   36.30%
1  C-only   220395         124369          24243                   71783             56.43%                   32.57%
2    both    46912          26891           6604                   13417             57.32%                   28.60%


---

**What this closes for Gap D.** Sections 11a-11d give the R/C source attribution that was lost at the blocking-stage dedup. Two operational implications:

1. **The `both` partition is the only one that can produce an R↔C link.** A name that appears `R-only` (e.g., an RC petition for a firm that has never had a CA charge) has no chance of being matched no matter how good clustering is. Same for `C-only` names. **The denominator for any R↔C matching-recall claim should be the `both` partition, not the full 334,276.**
2. **The Step-4 sampling frame should be drawn from the full `R ∪ C` universe**, with probability proportional to size (= name frequency in the address tables). The 3×3 table from §11b tells us how to stratify the draw: oversample `R-only` and `C-only` to ensure those populations are characterized, since they are jointly larger than `both` but cannot contribute to the matching evaluation.

These numbers belong in `name_data_flow.md` §5 as a new sub-section §5.4.
